# 04 — Assumptions

You have a baseline scenario. Now you want to ask "what if?" — what if Building B's heat pump improves from COP 3.2 to 4.0? What if everyone reduces electricity demand by 20% by 2035?

The `backend.assumptions` module does three flavours of this:

| Flavour | Function | What it does |
|---|---|---|
| **Single** | `apply_single_assumption` | One target component type + one attribute, one modification. Derives a single new scenario. |
| **Series** | `apply_series_assumption` | Same, but applied cumulatively across a list of timesteps. Derives N scenarios. |
| **Manual** | `apply_manual_modifications` | Specific attribute changes on a specific component (the UI's "edit this thing directly" mode). |

All three return new scenarios in the same dict shape the baseline uses, so they chain naturally.

## 4.1 Load the baseline

We read the baseline saved by notebook 03 (or fall back to fetching fresh from GraphDB).

In [ ]:
import os, sys, json, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))
os.environ.setdefault("TRIPLESTORE_BACKEND", "fuseki")
os.environ.setdefault("GRAPHDB_URL", "http://localhost:3030")

baseline_path = pathlib.Path("sample_data/alpine_village_baseline.json")
if not baseline_path.exists():
    raise FileNotFoundError(
        "Run 03_scenario_builder.ipynb first — it writes alpine_village_baseline.json"
    )
baseline = json.loads(baseline_path.read_text())
print(f"Loaded '{baseline['scenario_name']}' with {len(baseline['components'])} components")

## 4.2 Single assumption — efficiency retrofit

Ask: *"what if every energy consumer reduced their annual electricity demand by 20% thanks to a retrofit?"*

An assumption is a dict with four required keys:

| Key | Example | Meaning |
|---|---|---|
| `target_component` | `"EnergyConsumer"` | Apply only to components of this type |
| `target_attribute` | `"Annual electricity demand"` | Name of the attribute to modify |
| `modifier` | `"*"`, `"+"`, `"-"`, `"set"` | How to combine with the old value |
| `modifier_value` | `0.8` | The new-value operand |


In [ ]:
from backend.assumptions.assumption_engine import apply_single_assumption

retrofit = {
    "target_component": "EnergyConsumer",
    "target_attribute": "Annual electricity demand",
    "modifier":         "*",
    "modifier_value":   0.8,
}

scenario_retrofit = apply_single_assumption(
    baseline,
    retrofit,
    new_scenario_name="alpine_village_retrofit_2035",
    namespace="https://digicities.info/tutorial/alpine_village/scenarios/retrofit",
)

print(f"Modified {scenario_retrofit['modified_count']} components")
for entry in scenario_retrofit["modification_log"]:
    print(f"  {entry['component']:30}  {entry['old_value']:>8.0f}  →  {entry['new_value']:>8.0f}  ({entry['modification_type']})")

## 4.3 Series assumption — phased rollout

*"What if we phase in a 5% improvement every year from 2025 to 2030?"*

Series assumptions stack cumulatively. The modifier values list must be JSON-encoded, as must the timesteps — this is because the Streamlit UI stores them as text in session state.

In [ ]:
from backend.assumptions.assumption_engine import apply_series_assumption

phased = {
    "target_component":         "EnergyConsumer",
    "target_attribute":         "Annual electricity demand",
    "modifier":                 "*",
    "assumption_timesteps":     json.dumps(["2025", "2026", "2027", "2028", "2029", "2030"]),
    "modifier_value_series":    json.dumps([0.95, 0.95, 0.95, 0.95, 0.95, 0.95]),
}

series = apply_series_assumption(
    baseline,
    phased,
    base_scenario_name="alpine_village_phased",
    namespace="https://digicities.info/tutorial/alpine_village/scenarios/phased",
)

print(f"Generated {len(series)} scenarios:")
for s in series:
    # Grab Building B's demand as a representative component
    b = next((c for c in s['components'] if c['label'].startswith('Building B')), None)
    if b:
        demand = b['attributes'].get('Annual electricity demand', {}).get('value')
        print(f"  {s['timestep']}: Building B electricity demand = {float(demand):.0f} kWh")

## 4.4 Manual modification — specific tweak

*"Building B's heat pump improves from COP 3.2 to 4.0 after a firmware update."*

This targets a specific URI rather than "all components of type X". The modifications dict maps attribute-name → `{old_value, new_value, …}`.

In [ ]:
from backend.assumptions.manual_modification_engine import apply_manual_modifications

heat_pump_uri = next(
    c['uri'] for c in baseline['components'] if c['label'].startswith('Air-source heat pump')
)

firmware_upgrade = {
    "Seasonal COP": {
        "old_value": "3.2",
        "new_value": 4.0,
    },
}

tweaked = apply_manual_modifications(
    baseline,
    target_component_uri=heat_pump_uri,
    modifications=firmware_upgrade,
    new_scenario_name="alpine_village_firmware_upgrade",
    namespace="https://digicities.info/tutorial/alpine_village/scenarios/firmware",
)

for entry in tweaked['modification_log']:
    print(f"{entry['component']}: {entry['attribute']} {entry['old_value']} → {entry['new_value']} ({entry['change_percent']:+.1f}%)")

## 4.5 The contract between scenarios

Note what the engines return is structurally identical to the baseline they consumed:

- same `components` + `attributes` shape
- new URIs derived from `namespace`
- a `modification_log` carrying before/after values
- the original assumption dict embedded for traceability

This means you can chain them — apply a retrofit assumption, then a firmware tweak on top of *that* — without any glue code. Just feed one engine's output into another's input.

In [ ]:
# Chain: retrofit → then firmware upgrade on top
chained = apply_manual_modifications(
    scenario_retrofit,   # <-- feed the retrofit output back in
    target_component_uri=next(
        c['uri'] for c in scenario_retrofit['components']
        if c['label'].startswith('Air-source heat pump')
    ),
    modifications=firmware_upgrade,
    new_scenario_name="alpine_village_retrofit_plus_firmware_2035",
    namespace="https://digicities.info/tutorial/alpine_village/scenarios/combo",
)

print(f"Chained scenario has {len(chained['components'])} components and "
      f"{chained['modified_count']} applied modifications on top of the retrofit baseline.")

## Next

[`05_data_products.ipynb`](05_data_products.ipynb) — the TTL parser is what translates raw Turtle back into the component dicts you've just been working with.